# Roku ECP — Control Test Notebook

Roku's **External Control Protocol (ECP)** is a plain HTTP REST API on port `8060`.
No SDK, no auth, no pairing — just `requests` over your local network.

## Find your Roku IP
Settings → Network → About → IP address

In [ ]:
import requests
import xml.etree.ElementTree as ET
import json
import time
from urllib.parse import quote

ROKU_IP = "192.168.1.XXX"  # <-- replace with your Roku's IP
BASE    = f"http://{ROKU_IP}:8060"

def get(path):
    r = requests.get(f"{BASE}{path}", timeout=5)
    r.raise_for_status()
    return r

def post(path):
    r = requests.post(f"{BASE}{path}", timeout=5)
    r.raise_for_status()
    return r

print("Helpers ready.")

## 1 — Device info
Confirms the connection is live and shows model / software version.

In [ ]:
r = get("/")
root = ET.fromstring(r.text)

fields = ["friendly-device-name", "model-name", "software-version", "serial-number"]
for f in fields:
    el = root.find(f)
    print(f"{f:30s} {el.text if el is not None else '—'}")

## 2 — Installed apps
Returns every channel on the device with its numeric ID.
We need the IDs to launch Netflix and YouTube.

In [ ]:
r = get("/query/apps")
root = ET.fromstring(r.text)

apps = [(app.get("id"), app.text) for app in root.findall("app")]
apps.sort(key=lambda x: x[1].lower())

print(f"{'ID':>8}  Name")
print("-" * 40)
for app_id, name in apps:
    print(f"{app_id:>8}  {name}")

## 3 — Currently active app

In [ ]:
r = get("/query/active-app")
root = ET.fromstring(r.text)

app = root.find("app")
screensaver = root.find("screensaver")

if app is not None:
    print(f"Active app : {app.text} (id={app.get('id')})")
if screensaver is not None:
    print(f"Screensaver: {screensaver.text}")

## 4 — Launch apps
Standard Roku channel IDs:

| App | ID |
|---|---|
| Netflix | `12` |
| YouTube | `2285` |
| Disney+ | `291097` |
| Hulu | `2285` |
| Prime Video | `13` |

In [ ]:
APP_IDS = {
    "netflix": "12",
    "youtube": "2285",
}

def launch_app(name: str):
    app_id = APP_IDS[name.lower()]
    post(f"/launch/{app_id}")
    print(f"Launched {name} (id={app_id})")

launch_app("netflix")

## 5 — Search and play a show
Roku ECP search can launch straight into a title.
`launch=true` opens the title detail page immediately.

In [ ]:
def search_and_play(keyword: str, content_type: str = "series", launch: bool = True):
    """
    content_type: 'movie' | 'series' | 'person' | 'channel' | 'game'
    """
    params = [
        f"keyword={quote(keyword)}",
        f"type={content_type}",
        f"launch={'true' if launch else 'false'}",
    ]
    path = "/search/browse?" + "&".join(params)
    post(path)
    print(f"Searching for '{keyword}' ({content_type})")

search_and_play("Stranger Things", content_type="series")

## 6 — Playback controls
All standard Roku keypresses via `POST /keypress/<key>`.

In [ ]:
# Available keys
KEYS = {
    "play_pause" : "Play",
    "rewind"     : "Rev",
    "fast_fwd"   : "Fwd",
    "replay"     : "InstantReplay",   # 20-second instant replay
    "home"       : "Home",
    "back"       : "Back",
    "up"         : "Up",
    "down"       : "Down",
    "left"       : "Left",
    "right"      : "Right",
    "select"     : "Select",
    "vol_up"     : "VolumeUp",
    "vol_down"   : "VolumeDown",
    "mute"       : "VolumeMute",
}

def keypress(key: str):
    post(f"/keypress/{key}")
    print(f"Sent: {key}")

# Test — toggle play/pause
keypress(KEYS["play_pause"])

## 7 — Intent-to-keypress mapping
This mirrors what Gemma will output — a structured intent that maps to a Roku action.
Dementia-friendly language → button logic, invisibly.

In [ ]:
def handle_intent(intent: dict):
    """
    Accepts a structured intent dict (as Gemma will produce) and
    executes the corresponding Roku ECP action.

    Supported intents:
      { action: 'play' }
      { action: 'pause' }
      { action: 'rewind', amount: 'small' | 'large' }
      { action: 'forward', amount: 'small' | 'large' }
      { action: 'replay' }                            # 20-second instant replay
      { action: 'launch_show', title: str, app: str }
      { action: 'home' }
    """
    action = intent.get("action")
    narration = "Done."

    if action == "play":
        keypress("Play")
        narration = "Playing again. You're back where you left off."

    elif action == "pause":
        keypress("Play")  # Roku Play key toggles
        narration = "Paused. We can come back any time."

    elif action == "rewind":
        presses = 1 if intent.get("amount") == "small" else 3
        for _ in range(presses):
            keypress("Rev")
            time.sleep(0.3)
        keypress("Play")  # resume after seeking
        label = "ten seconds" if presses == 1 else "about thirty seconds"
        narration = f"I went back {label}. Nothing was lost."

    elif action == "forward":
        presses = 1 if intent.get("amount") == "small" else 3
        for _ in range(presses):
            keypress("Fwd")
            time.sleep(0.3)
        keypress("Play")
        label = "ten seconds" if presses == 1 else "about thirty seconds"
        narration = f"Skipped forward {label}."

    elif action == "replay":
        keypress("InstantReplay")
        narration = "I went back about twenty seconds so you can catch what you missed."

    elif action == "launch_show":
        title = intent.get("title", "")
        app   = intent.get("app", "netflix")
        launch_app(app)
        time.sleep(3)  # give the app time to open
        search_and_play(title)
        narration = f"Putting on {title}. Starting from where you left off."

    elif action == "home":
        keypress("Home")
        narration = "Taken back to your shows. Nothing was lost."

    print(f"Narration → {narration}")
    return narration


# --- Test intents ---

# "I missed that" → small rewind
handle_intent({ "action": "rewind", "amount": "small" })

In [ ]:
# "Go back a lot" → large rewind
handle_intent({ "action": "rewind", "amount": "large" })

In [ ]:
# "Put on Coronation Street"
handle_intent({ "action": "launch_show", "title": "Coronation Street", "app": "netflix" })

## 8 — Media player state
Returns what's currently playing: position, duration, buffering state.
Used to answer **"What episode is this?"** and to populate the UI.

In [ ]:
def get_media_state() -> dict:
    r = get("/query/media-player")
    root = ET.fromstring(r.text)

    state = {
        "state"    : root.get("state"),          # play | pause | stop | none
        "error"    : root.get("error"),
        "position" : None,
        "duration" : None,
        "live"     : None,
    }

    position = root.find("position")
    duration = root.find("duration")
    is_live  = root.find("is_live")

    if position is not None:
        state["position"] = position.text   # e.g. "00:12:34 (hh:mm:ss)"
    if duration is not None:
        state["duration"] = duration.text
    if is_live is not None:
        state["live"] = is_live.text == "true"

    return state

state = get_media_state()
print(json.dumps(state, indent=2))

## 9 — Full demo sequence
Simulates the complete Hearth TV flow:
1. Launch Netflix
2. Search and open a show
3. Check what's playing
4. Rewind a little
5. Pause
6. Go home

In [ ]:
def demo_sequence(show_title: str = "Stranger Things", app: str = "netflix"):
    print("=" * 50)
    print(f"DEMO: {show_title} on {app.title()}")
    print("=" * 50)

    print("\n[1] Launching app...")
    handle_intent({ "action": "launch_show", "title": show_title, "app": app })
    time.sleep(5)

    print("\n[2] Checking media state...")
    state = get_media_state()
    print(json.dumps(state, indent=2))

    print("\n[3] Rewinding a little ('I missed that')...")
    handle_intent({ "action": "replay" })
    time.sleep(2)

    print("\n[4] Pausing...")
    handle_intent({ "action": "pause" })
    time.sleep(2)

    print("\n[5] Going home...")
    handle_intent({ "action": "home" })

    print("\nDemo complete.")

# Uncomment to run:
# demo_sequence("Stranger Things", "netflix")